# Emissions Forecast Tool: Data Pipeline

**Author:** Yenlik Gaisina | Data & Analytics Consultant
**Data Source:** World Bank Open Data (CO2 emissions, GDP, population)
**Objective:** Download, clean, and analyse national CO2 emissions data; build ARIMA forecasts for selected countries

---

## 1. Data Download

The World Bank provides free API access to development indicators including CO2 emissions per capita, total CO2 emissions, GDP, and population. Data is downloaded programmatically and cached locally.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import os
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#e6edf3',
    'grid.color': '#21262d',
    'grid.alpha': 0.6,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'figure.dpi': 120,
})

print('Libraries loaded')

In [ ]:
# ── World Bank API: Download CO2 emissions data ───────────────────────
CACHE_PATH = '../data/clean/wb_emissions.csv'

INDICATORS = {
    'EN.ATM.CO2E.KT': 'co2_kt',          # CO2 emissions (kt)
    'EN.ATM.CO2E.PC': 'co2_per_capita',   # CO2 per capita (metric tons)
    'NY.GDP.MKTP.CD': 'gdp_usd',         # GDP (current US$)
    'SP.POP.TOTL': 'population',          # Population total
}

COUNTRIES = ['GBR', 'USA', 'CHN', 'IND', 'DEU', 'FRA', 'JPN', 'BRA', 'ZAF', 'AUS']

def fetch_wb_indicator(indicator, countries, start=1990, end=2022):
    """Download a single indicator from the World Bank API."""
    country_str = ';'.join(countries)
    url = f'https://api.worldbank.org/v2/country/{country_str}/indicator/{indicator}'
    params = {'format': 'json', 'per_page': 5000, 'date': f'{start}:{end}'}
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()
    if len(data) < 2:
        return pd.DataFrame()
    records = []
    for entry in data[1]:
        if entry['value'] is not None:
            records.append({
                'country': entry['country']['value'],
                'country_code': entry['countryiso3code'],
                'year': int(entry['date']),
                'value': float(entry['value']),
            })
    return pd.DataFrame(records)

if os.path.exists(CACHE_PATH):
    df = pd.read_csv(CACHE_PATH)
    print(f'Loaded from cache: {len(df):,} rows')
else:
    frames = []
    for indicator, col_name in INDICATORS.items():
        print(f'Downloading {col_name} ({indicator})...')
        tmp = fetch_wb_indicator(indicator, COUNTRIES)
        tmp = tmp.rename(columns={'value': col_name})
        frames.append(tmp)

    df = frames[0]
    for f in frames[1:]:
        df = df.merge(f[['country_code', 'year', f.columns[-1]]],
                       on=['country_code', 'year'], how='outer')

    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    df.to_csv(CACHE_PATH, index=False)
    print(f'Downloaded and cached: {len(df):,} rows')

print(f'\nShape: {df.shape}')
print(f'Countries: {df["country"].nunique()}')
print(f'Year range: {df["year"].min()} - {df["year"].max()}')
df.head()

## 2. Exploratory Analysis

In [ ]:
# ── CO2 emissions trends by country ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Total CO2 emissions
ax = axes[0]
for country in ['United Kingdom', 'United States', 'China', 'India', 'Germany']:
    subset = df[df['country'] == country].sort_values('year')
    ax.plot(subset['year'], subset['co2_kt'] / 1e6, linewidth=2, label=country)
ax.set_xlabel('Year')
ax.set_ylabel('CO2 Emissions (Gt)')
ax.set_title('Total CO2 Emissions by Country')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Per capita CO2
ax = axes[1]
for country in ['United Kingdom', 'United States', 'China', 'India', 'Germany']:
    subset = df[df['country'] == country].sort_values('year')
    ax.plot(subset['year'], subset['co2_per_capita'], linewidth=2, label=country)
ax.set_xlabel('Year')
ax.set_ylabel('CO2 per Capita (tonnes)')
ax.set_title('CO2 Emissions per Capita')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Carbon intensity: CO2 per unit GDP ────────────────────────────────
df['co2_per_gdp'] = df['co2_kt'] * 1000 / df['gdp_usd'] * 1e6  # tonnes CO2 per million USD GDP

fig, ax = plt.subplots(figsize=(12, 6))
for country in ['United Kingdom', 'United States', 'China', 'India', 'Germany', 'Japan']:
    subset = df[df['country'] == country].sort_values('year')
    ax.plot(subset['year'], subset['co2_per_gdp'], linewidth=2, label=country)
ax.set_xlabel('Year')
ax.set_ylabel('Carbon Intensity (t CO2 per $M GDP)')
ax.set_title('Carbon Intensity of GDP')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. UK Emissions Forecast

ARIMA and ETS models are applied to the UK CO2 emissions time series to forecast 8 years ahead (to 2030).

In [ ]:
# ── UK CO2 time series ────────────────────────────────────────────────
uk = df[df['country_code'] == 'GBR'][['year', 'co2_kt']].dropna().sort_values('year')
uk = uk.set_index('year')
uk.index = pd.PeriodIndex(uk.index, freq='Y')

# ADF test
result = adfuller(uk['co2_kt'].dropna())
print(f'ADF Statistic: {result[0]:.4f}')
print(f'p-value: {result[1]:.4f}')
print(f'Stationary: {"YES" if result[1] < 0.05 else "NO - differencing required"}')
print(f'\nUK CO2 series: {len(uk)} observations ({uk.index[0]} to {uk.index[-1]})')

In [ ]:
# ── ARIMA Forecast ─────────────────────────────────────────────────────
model = SARIMAX(uk['co2_kt'], order=(1, 1, 1), enforce_stationarity=False)
fit = model.fit(disp=False)
print(fit.summary().tables[0])

forecast_steps = 8
forecast = fit.get_forecast(steps=forecast_steps)
fc_mean = forecast.predicted_mean
fc_ci = forecast.conf_int(alpha=0.10)

# ETS comparison
ets = ExponentialSmoothing(uk['co2_kt'], trend='add', seasonal=None).fit()
ets_forecast = ets.forecast(forecast_steps)

# Plot
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(uk.index.to_timestamp(), uk['co2_kt'] / 1e3, color='#388bfd', lw=2, label='Observed')
fc_dates = pd.period_range(uk.index[-1] + 1, periods=forecast_steps, freq='Y').to_timestamp()
ax.plot(fc_dates, fc_mean.values / 1e3, color='#238636', lw=2.5, ls='--', marker='o', ms=4, label='ARIMA(1,1,1)')
ax.fill_between(fc_dates, fc_ci.iloc[:, 0].values / 1e3, fc_ci.iloc[:, 1].values / 1e3,
                alpha=0.2, color='#238636', label='90% CI')
ax.plot(fc_dates, ets_forecast.values / 1e3, color='#e3b341', lw=2, ls=':', marker='s', ms=4, label='ETS')
ax.axvline(uk.index[-1].to_timestamp(), color='#8b949e', ls=':', alpha=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('CO2 Emissions (Mt)')
ax.set_title('UK CO2 Emissions Forecast to 2030')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'UK CO2 {uk.index[-1]}: {uk["co2_kt"].iloc[-1]/1e3:.1f} Mt')
print(f'ARIMA 2030 forecast: {fc_mean.values[-1]/1e3:.1f} Mt')
print(f'ETS 2030 forecast: {ets_forecast.values[-1]/1e3:.1f} Mt')

## 4. Multi-Country Comparison

In [ ]:
# ── Latest year comparison ─────────────────────────────────────────────
latest = df.groupby('country').apply(
    lambda x: x.dropna(subset=['co2_kt', 'co2_per_capita', 'gdp_usd']).iloc[-1]
    if len(x.dropna(subset=['co2_kt', 'co2_per_capita', 'gdp_usd'])) > 0
    else None
).dropna()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Total emissions bar
ax = axes[0]
data = latest.sort_values('co2_kt', ascending=True)
ax.barh(data['country'], data['co2_kt'] / 1e6, color='#388bfd', alpha=0.8)
ax.set_xlabel('CO2 (Gt)')
ax.set_title('Total CO2 Emissions')
ax.grid(True, alpha=0.3, axis='x')

# Per capita bar
ax = axes[1]
data2 = latest.sort_values('co2_per_capita', ascending=True)
ax.barh(data2['country'], data2['co2_per_capita'], color='#f85149', alpha=0.8)
ax.set_xlabel('Tonnes per capita')
ax.set_title('CO2 per Capita')
ax.grid(True, alpha=0.3, axis='x')

# Carbon intensity
ax = axes[2]
latest['intensity'] = latest['co2_kt'] * 1000 / latest['gdp_usd'] * 1e6
data3 = latest.sort_values('intensity', ascending=True)
ax.barh(data3['country'], data3['intensity'], color='#e3b341', alpha=0.8)
ax.set_xlabel('t CO2 per $M GDP')
ax.set_title('Carbon Intensity')
ax.grid(True, alpha=0.3, axis='x')

fig.suptitle('Global Emissions Comparison (Latest Available Year)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Summary

This notebook demonstrates the data pipeline for the Emissions Forecast Tool:

1. **Data download**: World Bank API provides free, programmatic access to CO2 emissions, GDP, and population data for 10 countries (1990-2022)
2. **Exploratory analysis**: Trends, per-capita comparisons, and carbon intensity metrics
3. **Forecasting**: ARIMA and ETS models forecast UK emissions to 2030
4. **The accompanying Streamlit app** (`app/app.py`) provides an interactive interface for selecting countries, viewing trends, and generating forecasts